# Medallion Pipeline - Business Intelligence Dashboard
Interactive visualizations powered by Gold Layer aggregations (`dev_gold.reporting`)

In [0]:
%pip install plotly kaleido --quiet

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

## Executive KPIs

In [0]:
df_kpi = spark.sql("""
    SELECT 
        ROUND(SUM(total_revenue), 0) as total_revenue,
        ROUND(SUM(total_profit), 0) as total_profit,
        SUM(total_orders) as total_orders,
        ROUND(AVG(avg_profit_margin) * 100, 1) as avg_margin_pct
    FROM dev_gold.reporting.agg_monthly_sales
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Indicator(mode="number", value=df_kpi['total_revenue'][0], title={"text": "Total Revenue ($)"}, domain={'x': [0, 0.25], 'y': [0, 1]}, number={'prefix': '$', 'valueformat': ',.0f'}))
fig.add_trace(go.Indicator(mode="number", value=df_kpi['total_profit'][0], title={"text": "Total Profit ($)"}, domain={'x': [0.25, 0.5], 'y': [0, 1]}, number={'prefix': '$', 'valueformat': ',.0f'}))
fig.add_trace(go.Indicator(mode="number", value=df_kpi['total_orders'][0], title={"text": "Total Orders"}, domain={'x': [0.5, 0.75], 'y': [0, 1]}, number={'valueformat': ',.0f'}))
fig.add_trace(go.Indicator(mode="number", value=df_kpi['avg_margin_pct'][0], title={"text": "Avg Profit Margin"}, domain={'x': [0.75, 1.0], 'y': [0, 1]}, number={'suffix': '%', 'valueformat': '.1f'}))
fig.update_layout(height=200, margin=dict(t=40, b=20, l=20, r=20), paper_bgcolor='white')
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Monthly Revenue & Profit Trends

In [0]:
df_monthly = spark.sql("""
    SELECT CAST(year AS STRING) as year, month, month_name, 
           ROUND(total_revenue, 2) as revenue, 
           ROUND(total_profit, 2) as profit,
           total_orders
    FROM dev_gold.reporting.agg_monthly_sales 
    ORDER BY year, month
""").toPandas()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Monthly Revenue by Year", "Monthly Profit by Year"))

for year in sorted(df_monthly['year'].unique()):
    yr_data = df_monthly[df_monthly['year'] == year]
    fig.add_trace(go.Scatter(x=yr_data['month'], y=yr_data['revenue'], name=f'{year} Revenue', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(x=yr_data['month'], y=yr_data['profit'], name=f'{year} Profit', mode='lines+markers'), row=1, col=2)

fig.update_layout(height=400, showlegend=True, margin=dict(t=50, b=40))
fig.update_xaxes(title_text="Month", dtick=1)
fig.update_yaxes(title_text="Revenue ($)", row=1, col=1)
fig.update_yaxes(title_text="Profit ($)", row=1, col=2)
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Regional Performance & Customer Segments

In [0]:
df_region = spark.sql("""
    SELECT region, manager_name, 
           ROUND(total_revenue, 0) as revenue, 
           ROUND(total_profit, 0) as profit,
           total_customers,
           ROUND(revenue_per_customer, 0) as rev_per_customer
    FROM dev_gold.reporting.agg_regional_summary 
    ORDER BY revenue DESC
""").toPandas()

df_segments = spark.sql("""
    SELECT segment, COUNT(*) as customer_count
    FROM dev_gold.reporting.agg_customer_lifetime_value 
    GROUP BY segment ORDER BY customer_count DESC
""").toPandas()

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "bar"}, {"type": "pie"}]], subplot_titles=("Revenue by Region", "Customer Segments"))
fig.add_trace(go.Bar(x=df_region['region'], y=df_region['revenue'], marker_color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'], text=df_region['revenue'].apply(lambda x: f'${x:,.0f}'), textposition='outside'), row=1, col=1)
fig.add_trace(go.Pie(labels=df_segments['segment'], values=df_segments['customer_count'], hole=0.4, textinfo='label+percent'), row=1, col=2)
fig.update_layout(height=400, showlegend=False, margin=dict(t=50, b=40))
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Top 10 Products by Revenue

In [0]:
df_products = spark.sql("""
    SELECT product_name, category, 
           ROUND(total_revenue, 0) as revenue,
           ROUND(total_profit, 0) as profit,
           total_units_sold,
           ROUND(return_rate * 100, 1) as return_rate_pct
    FROM dev_gold.reporting.agg_product_performance 
    ORDER BY revenue DESC LIMIT 10
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Bar(
    y=df_products['product_name'], 
    x=df_products['revenue'],
    orientation='h',
    marker_color=df_products['category'].map({'Technology': '#2196F3', 'Furniture': '#4CAF50', 'Office Supplies': '#FF9800'}),
    text=df_products['revenue'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside'
))
fig.update_layout(title="Top 10 Products by Revenue", height=500, margin=dict(l=300, t=50, b=40), xaxis_title="Revenue ($)", yaxis={'categoryorder': 'total ascending'})
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Category Trends (Quarterly)

In [0]:
df_cat = spark.sql("""
    SELECT CAST(year AS STRING) || '-Q' || CAST(quarter AS STRING) as period,
           category, ROUND(total_revenue, 0) as revenue
    FROM dev_gold.reporting.agg_category_trends 
    ORDER BY year, quarter
""").toPandas()

fig = px.line(df_cat, x='period', y='revenue', color='category', markers=True,
              title="Category Revenue Trends (Quarterly)", labels={'revenue': 'Revenue ($)', 'period': 'Period'})
fig.update_layout(height=400, margin=dict(t=50, b=40))
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Customer Lifetime Value Analysis

In [0]:
df_ltv = spark.sql("""
    SELECT ltv_tier, COUNT(*) as customer_count, 
           ROUND(AVG(lifetime_revenue), 0) as avg_revenue,
           ROUND(AVG(total_orders), 1) as avg_orders
    FROM dev_gold.reporting.agg_customer_lifetime_value 
    GROUP BY ltv_tier ORDER BY avg_revenue DESC
""").toPandas()

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]], subplot_titles=("LTV Tier Distribution", "Avg Revenue by LTV Tier"))
fig.add_trace(go.Pie(labels=df_ltv['ltv_tier'], values=df_ltv['customer_count'], hole=0.3, textinfo='label+percent', marker=dict(colors=['#FFD700', '#C0C0C0', '#CD7F32', '#E5E4E2'])), row=1, col=1)
fig.add_trace(go.Bar(x=df_ltv['ltv_tier'], y=df_ltv['avg_revenue'], marker_color=['#FFD700', '#C0C0C0', '#CD7F32', '#E5E4E2'], text=df_ltv['avg_revenue'].apply(lambda x: f'${x:,.0f}'), textposition='outside'), row=1, col=2)
fig.update_layout(height=400, showlegend=False, margin=dict(t=50, b=40))
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Shipping Analysis

In [0]:
df_ship = spark.sql("""
    SELECT ship_mode, ROUND(total_revenue, 0) as revenue, total_orders,
           ROUND(avg_delivery_days, 1) as avg_days,
           ROUND(avg_profit_margin * 100, 1) as margin_pct,
           ROUND(revenue_share_pct, 1) as rev_share
    FROM dev_gold.reporting.agg_shipping_analysis ORDER BY revenue DESC
""").toPandas()

fig = make_subplots(rows=1, cols=3, specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "xy"}]], subplot_titles=("Revenue Share", "Avg Delivery Days", "Profit Margin %"))
fig.add_trace(go.Pie(labels=df_ship['ship_mode'], values=df_ship['rev_share'], hole=0.4, textinfo='label+percent'), row=1, col=1)
fig.add_trace(go.Bar(x=df_ship['ship_mode'], y=df_ship['avg_days'], marker_color=['#F44336', '#FF9800', '#4CAF50', '#2196F3'], text=df_ship['avg_days'].apply(lambda x: f'{x:.1f}d'), textposition='outside'), row=1, col=2)
fig.add_trace(go.Bar(x=df_ship['ship_mode'], y=df_ship['margin_pct'], marker_color=['#F44336', '#FF9800', '#4CAF50', '#2196F3'], text=df_ship['margin_pct'].apply(lambda x: f'{x:.1f}%'), textposition='outside'), row=1, col=3)
fig.update_layout(height=400, showlegend=False, margin=dict(t=50, b=40))
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Monthly Order Volume

In [0]:
fig = go.Figure()
for year in sorted(df_monthly['year'].unique()):
    yr_data = df_monthly[df_monthly['year'] == year]
    fig.add_trace(go.Bar(x=yr_data['month'], y=yr_data['total_orders'], name=f'{year}'))

fig.update_layout(title="Monthly Order Volume by Year", barmode='group', height=400, xaxis_title="Month", yaxis_title="Orders", margin=dict(t=50, b=40))
displayHTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

## Regional Manager Performance

In [0]:
df_mgr = spark.sql("""
    SELECT manager_name, region, 
           ROUND(total_revenue, 0) as revenue,
           ROUND(total_profit, 0) as profit,
           total_customers, total_orders,
           ROUND(avg_profit_margin * 100, 1) as margin_pct
    FROM dev_gold.reporting.agg_regional_summary ORDER BY revenue DESC
""").toPandas()

displayHTML(df_mgr.to_html(index=False, classes='table table-striped', border=0, justify='left'))

## Top Customers by Lifetime Value

In [0]:
df_top_cust = spark.sql("""
    SELECT customer_name, segment, ltv_tier,
           ROUND(lifetime_revenue, 0) as lifetime_revenue,
           total_orders, ROUND(avg_order_value, 0) as avg_order
    FROM dev_gold.reporting.agg_customer_lifetime_value 
    ORDER BY lifetime_revenue DESC LIMIT 15
""").toPandas()

displayHTML(df_top_cust.to_html(index=False, classes='table table-striped', border=0, justify='left'))